# CSV vs Parquet load speed comparison
Using `matrix_pfe_mapq30` as the test case (~2.6 GB CSV, 459 samples × 561k features).

In [1]:
import pandas as pd
import numpy as np
import time
import os
os.chdir("/data/projects/liquid_biopsy/Projects/cfDNA/cfDNA/")
CSV_PATH     = './data/matrix/by_feature/matrix_pfe_mapq30.csv'
PARQUET_PATH = './data/matrix/by_feature/matrix_pfe_mapq30.parquet'
METADATA_COLS = ['sample_id', 'disease', 'dataset', 'material', 'stage', 'cancer_true']

## Step 1 — Convert CSV → Parquet (one-time)

In [8]:
if os.path.exists(PARQUET_PATH):
    print(f"Parquet already exists: {PARQUET_PATH}")
else:
    print("Converting CSV → Parquet (this runs once, takes a while) ...")
    t0 = time.perf_counter()
    df = pd.read_csv(CSV_PATH, index_col=0, low_memory=False)
    numeric_cols = [c for c in df.columns if c not in METADATA_COLS]
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
    df.to_parquet(PARQUET_PATH)
    elapsed = time.perf_counter() - t0
    size_mb = os.path.getsize(PARQUET_PATH) / 1024**2
    print(f"Done in {elapsed:.1f}s  →  {size_mb:.0f} MB on disk")

csv_mb     = os.path.getsize(CSV_PATH)     / 1024**2
parquet_mb = os.path.getsize(PARQUET_PATH) / 1024**2
print(f"\nCSV size:     {csv_mb:.0f} MB")
print(f"Parquet size: {parquet_mb:.0f} MB  ({parquet_mb/csv_mb*100:.1f}% of CSV)")

Converting CSV → Parquet (this runs once, takes a while) ...


KeyboardInterrupt: 

## Step 2 — Time CSV load

In [ ]:
t0 = time.perf_counter()
df_csv = pd.read_csv(CSV_PATH, index_col=0, low_memory=False)
numeric_cols = [c for c in df_csv.columns if c not in METADATA_COLS]
df_csv[numeric_cols] = df_csv[numeric_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
csv_time = time.perf_counter() - t0
print(f"CSV load + cast:  {csv_time:.1f}s   shape={df_csv.shape}")

## Step 3 — Time Parquet load

In [3]:
t0 = time.perf_counter()
df_parquet = pd.read_parquet(PARQUET_PATH)
parquet_time = time.perf_counter() - t0
print(f"Parquet load:     {parquet_time:.1f}s   shape={df_parquet.shape}")

FileNotFoundError: [Errno 2] No such file or directory: './data/matrix/by_feature/matrix_pfe_mapq30.parquet'

## Step 4 — Summary

In [ ]:
speedup = csv_time / parquet_time
print(f"CSV load time:     {csv_time:.1f}s")
print(f"Parquet load time: {parquet_time:.1f}s")
print(f"Speedup:           {speedup:.1f}x")
print(f"\nDtypes match: {(df_csv[numeric_cols].dtypes == df_parquet[numeric_cols].dtypes).all()}")
print(f"Values match:  {np.allclose(df_csv[numeric_cols].values, df_parquet[numeric_cols].values, equal_nan=True)}")